<a href="https://colab.research.google.com/github/praksb2428-maker/Deep.practice/blob/main/project_mid_report_md.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 딥러닝 기반 여름철 기온 예측 및 폭염 확률 분석 프로젝트
## 중간 보고서: 기획안 대비 주요 개선점 및 모델 고도화 성과

**일자:** 2026년 5월 17일  
**개발자:** 박준호  
**프로젝트 단계:** 중간 점검 및 핵심 알고리즘 고도화 완료  

## 1. 프로젝트 개요
본 프로젝트는 서울 지역의 과거 기상청 관측 데이터(기온, 강수량 등)를 다각도로 전처리하여, 여름철(6월 ~ 8월)의 일 평균기온, 일 최고기온 및 폭염 발생 확률을 정밀하게 예측하는 딥러닝 모델을 구축하는 것을 목적으로 합니다.

현재 작성중인 모델에서 마주한 시계열 데이터의 한계점과 미래 예측 공백 문제를 엔지니어링 기법으로 보완하였으며, 본 중간 보고서는 기획 대비 모델링 과정에서 달성한 실전적 개선 성과를 종합적으로 다룹니다.


## 2. 핵심 기술적 개선점 상세 분석 (Key Improvements)

### 개선 1: 버퍼(Buffer) 데이터셋 기법 도입을 통한 경계선 문제 해결
* **기존 문제점:** 최초 기획대로 6월-8월 여름철 데이터만 잘라내어 시계열 윈도우를 구성할 경우, 6월 1일-15일 사이의 초순 날짜를 예측할 때 직전 15일치 데이터(5월 중하순)가 존재하지 않는 '물리적 데이터 단절'이 발생했습니다. 이로 인해 여름 데이터의 약 15%를 시작부터 손실하거나, 연도가 넘어갈 때 '작년 8월 말 날씨' 직후에 '올해 6월 초 날씨'가 강제 결합되는 데이터 오염(Time Leak) 문제가 발생했습니다.
* **해결 방안:** 원천 데이터 필터링 단계에서 **5월 데이터를 '힌트용 버퍼(Buffer)'로 명시적으로 포함**시켰습니다.
* **성과:** `create_summer_dataset` 함수를 설계하여 5월 데이터는 오직 과거 15일치 추세(`X_data`)로만 활용되고, 인공지능이 풀고 채점받는 정답지(`y_data`)에는 순수 6, 7, 8월만 담기도록 격리했습니다. 이로써 단 하루의 데이터 손실도 없이 6월 1일부터 완벽하게 추론하는 여름 특화 모델이 완성되었습니다.

###개선 2: 무제한 미래 예측 확장 및 지구온난화 연간 트렌드 가중치 반영
* **기존 문제점:** 최초 코드에서는 `base_year = 2024` 등으로 과거 연도가 강제 고정되어 있었습니다. 이 때문에 사용자가 2025년을 입력하든 2026년을 입력하든 내부적으로는 계속 똑같은 2024년 윈도우를 읽어와 **소수점까지 완벽히 일치하는 의미 없는 예측 결과가 반복**되었습니다. 또한 데이터셋이 2025년에서 끝나기 때문에 2027년 이후를 입력하면 데이터가 없어 터지는 치명적인 버그가 있었습니다.
* **해결 방안:** 1. **동적 연도 매핑 및 안전 고정(Clamping):** 사용자가 입력한 연도의 직전 해(`target_date.year - 1`)를 자동 추적하게 하되, 데이터셋 한계선(2025년)을 초과하는 미래 연도(2027년 이상)를 입력하면 가장 최신인 2025년 기상 시퀀스를 대리 시험지로 쓰도록 안전장치를 쳤습니다.
  2. **장기 기후 트렌드 가중치(`TREND_PER_YEAR = 0.15℃`) 도입:** 단순히 과거 데이터에만 의존하지 않고, 2026년을 기준으로 미래 연도가 1년씩 증가할 때마다 지구온난화 및 기후 변화 추이를 반영하여 평균/최고기온이 누적 가산되도록 수학적으로 결합했습니다.
* **성과:** 2027년, 2028년 등 먼 미래를 입력해도 프로그램이 터지지 않고, 연도 격차에 비례해 누적 가산치(`+0.15℃`, `+0.30℃` 등)가 유동적으로 더해진 다이나믹한 미래 예측 대시보드가 완성되었습니다.

### 개선 3: 기획서 기반 강수량 40mm 이상 사후 보정 알고리즘 이식
* **기존 문제점:** 기획서상에는 강수량 수준에 따른 가중치 부여 체감 온도 보정 규칙(+2.0℃ 가산)이 존재했으나, 원본 파이썬 소스코드 내부에는 단순히 2mm 이상일 때 비 경고 메시지만 띄울 뿐 실제 기온 연산과 연동되어 있지 않았습니다.
* **해결 방안:** 대화형 예측 함수 내부에서 사용자가 지정한 날짜의 과거 관측 강수량을 직접 조회하게 했습니다. 만약 **당일 강수량이 40mm 이상의 집중호우 조건에 부합할 경우 강수 라벨을 1로 표기하고, 인공지능이 예측한 평균기온과 최고기온 결과값에 각각 2.0℃를 강제로 더해주는 사후 보정 조건문**을 완벽히 구축했습니다.
* **성과:** 인공지능의 수치 예측 한계를 도메인 지식(Rule-based weight)으로 보완하여, 비가 올 때의 체감 온도 변화 및 가이드라인을 정교하게 출력할 수 있게 되었습니다.

### 개선 4: 시계열 정석 교차검증(TimeSeriesSplit) 및 4대 평가지표 시각화 고도화
* **기존 문제점:** 일반적인 데이터 쪼개기(Train_Test_Split) 방식은 미래 데이터가 과거 모델 학습에 섞여 들어가는 타임 리크 오류를 범하기 쉽고, 단순 수치 출력만으로는 모델의 안정성을 검증하기 어려웠습니다.
* **해결 방안:** 데이터의 시간 순서를 철저히 보존하면서 점진적으로 학습 데이터셋을 늘려나가는 **`TimeSeriesSplit(n_splits=5)` 교차검증**을 수행했습니다. 이와 동시에 최신 연도 데이터일수록 가중치를 점진적으로 높게 설정하는 `sample_weight`를 적용했습니다 (2024년 기준 최대 3.5배 우대).
* **성과:** 각 폴드별로 **MAE, RMSE, R², MAPE**라는 4대 평가지표를 평균기온과 최고기온에 대해 각각 정밀 산출하였고, 교차검증 완료 시 5개 폴드의 성적 변화 안정성을 한눈에 모니터링할 수 있는 **2x2 격자형 선형 추이 시각화 그래프(Stability Track)** 코드를 완전히 내장하여 분석 완성도를 끌어올렸습니다.

## 3. 기획안 대비 변경 및 제외 항목 (Exclusions)
* **GRU(Gate Recurrent Unit) 모델 구현 제외:** 최초 기획 시 LSTM과의 비교를 언급했으나, 프로젝트의 일관성과 하이브리드 신경망 아키텍처의 연산 집중도를 위해 `Conv1D + LSTM` 단일 구조를 고도화하는 방향으로 선회했습니다.
* **RobustScaler 고정:** 여러 스케일러 후보군 중 한국 여름철 특유의 기록적인 폭염 및 게릴라성 폭우 등 극단적 기후 이상치에 모델이 무너지지 않도록, 중앙값(Median)과 IQR을 사용하는 `RobustScaler`로 전처리 설계를 조기 확정했습니다.

## 4. 향후 일정 및 기대효과 (Future Plans)
* **기대효과:** 본 프로젝트를 통해 구축된 동적 시계열 예측 모델은 단순 기상 예보를 넘어, 사용자가 미래 특정 날짜를 입력했을 때 과거 기후 패턴과 장기 온난화 추이, 호우 시 체감 온도 보정치까지 종합 연산된 맞춤형 기록을 제공합니다. 이는 사용자가 데이터를 바탕으로 여름철 하루를 능동적으로 설계할 수 있도록 돕는 기술적 전환점이 될 것입니다.
* **최종 단계 계획:** 1. 현재 구축된 3단계 통합 파이프라인 코드의 주석 정밀화 및 Colab 환경 배포 안정성 테스트.
  1. 폭염일 가능성 분류 임계치(Threshold)의 정밀 튜닝을 통한 오분류율(False Positive) 개선.
  2. 중간 보고서 성과를 바탕으로 한 최종 포트폴리오 및 프로젝트 보고서 작성 완료.